In [ ]:
# ================================================================================
# STEP 1: Setup and Imports
# ================================================================================
# Import all required libraries and set up the environment

# Core CrewAI imports
from crewai import Agent, Crew, Task, Process, LLM
from crewai.tools import BaseTool
from crewai_tools import SerperDevTool

# Pydantic for structured outputs
from pydantic import BaseModel, Field
from typing import List, Type, Optional

# Environment and utilities
import os
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Load environment variables from .env file
load_dotenv()

# Verify API keys are set
if not os.getenv('OPENAI_API_KEY'):
    print("⚠️ Warning: OPENAI_API_KEY not found in environment")
if not os.getenv('SERPER_API_KEY'):
    print("⚠️ Warning: SERPER_API_KEY not found in environment")

print("✅ Libraries imported successfully!")

# Initialize the LLM
llm = LLM(model="gpt-4")
print("✅ LLM initialized (GPT-4)")

In [ ]:

# --- Structured Outputs with Pydantic models ---

class LearningMaterial(BaseModel):
    title: str
    url: Optional[str] = None
    type: str  # 'video', 'article', 'exercise'
    description: Optional[str] = None

class LearningMaterialsOutput(BaseModel):
    materials: List[LearningMaterial] = Field(description="A list of curated learning materials")

class QuizQuestion(BaseModel):
    question: str
    choices: List[str]
    answer: str

class QuizOutput(BaseModel):
    questions: List[QuizQuestion]

class ProjectIdea(BaseModel):
    title: str
    description: str
    expertise_level: str  # 'beginner', 'intermediate', 'advanced'

class ProjectSuggestionsOutput(BaseModel):
    projects: List[ProjectIdea]

# --- Custom Tool: ProjectSuggestionTool ---

class ProjectSuggestionInput(BaseModel):
    expertise_level: str
    topics: List[str]

class ProjectSuggestionTool(BaseTool):
    name: str = "Project Suggestion Tool"
    description: str = "Suggests practical project ideas based on expertise level and topics of interest."
    args_schema: Type[BaseModel] = ProjectSuggestionInput

    def _run(self, expertise_level: str, topics: List[str]) -> str:
        # Here you can integrate with a smart backend or just return a mock suggestion
        joined_topics = ', '.join(topics)
        if expertise_level.lower() == 'beginner':
            desc = f"A simple project for learning: Create a flashcard app to review {joined_topics}."
        elif expertise_level.lower() == 'intermediate':
            desc = f"An intermediate project: Build a blog engine that lets users post and quiz each other about {joined_topics}."
        else:
            desc = f"An advanced project: Implement a full-stack personalized learning platform specializing in {joined_topics} topics."
        idea = ProjectIdea(title=f"{expertise_level.title()} {joined_topics} Project", description=desc, expertise_level=expertise_level)
        return ProjectSuggestionsOutput(projects=[idea]).json()

# --- User Input Setup ---

USER_TOPICS = ["Python programming"]
USER_EXPERTISE = "intermediate"

# --- Agents ---

learning_material_agent = Agent(
    llm=LLM(model="gpt-4"),
    role="Learning Material Agent",
    backstory="Expert in curating engaging learning materials across subject domains using web search tools and best resources.",
    goal="Curate personalized learning materials (videos, articles, exercises) for users based on their topics of interest. Respond with a structured Pydantic output.",
    verbose=True,
    tools=[SerperDevTool()],
    output_schema=LearningMaterialsOutput,
)

quiz_creator_agent = Agent(
    llm=LLM(model="gpt-4"),
    role="Quiz Creator Agent",
    backstory="A creative quiz-maker who helps reinforce user learning by generating tailored questions.",
    goal="Generate appropriate quiz questions (multiple choice) to test understanding of learning materials. Structure responses as Pydantic models.",
    verbose=True,
    output_schema=QuizOutput
)

project_idea_agent = Agent(
    llm=LLM(model="gpt-4"),
    role="Project Idea Agent",
    backstory="Inventive assistant who suggests practical project ideas to cement learning and build real-world skills.",
    goal="Propose project ideas based on the user's expertise level and topics. Response is structured as a ProjectSuggestionsOutput Pydantic model.",
    verbose=True,
    tools=[ProjectSuggestionTool()],
    output_schema=ProjectSuggestionsOutput
)

# --- Tasks ---

learning_material_task = Task(
    description=(
        f"Curate a list of videos, articles, and exercises for the user on these topics: {', '.join(USER_TOPICS)}. "
        "Use web search tools to find a balance of engaging and authoritative resources. The output should be a list of LearningMaterial objects in a Pydantic LearningMaterialsOutput."
    ),
    agent=learning_material_agent,
    expected_output="A structured list of learning materials (videos, articles, exercises).",
    output_schema=LearningMaterialsOutput
)

quiz_task = Task(
    description=(
        "Given the curated learning materials, generate 5-7 quiz questions (multiple choice) to assess the user's understanding of the above topics. "
        "Return as a list of QuizQuestion objects using the QuizOutput schema."
    ),
    agent=quiz_creator_agent,
    expected_output="A structured quiz with questions, choices, and answers.",
    output_schema=QuizOutput
)

project_task = Task(
    description=(
        f"Suggest practical project ideas for a user whose expertise level is '{USER_EXPERTISE}' and whose topics of interest are {', '.join(USER_TOPICS)}. "
        "Project suggestions should help the user apply their knowledge. Use the Project Suggestion Tool. Respond with a ProjectSuggestionsOutput."
    ),
    agent=project_idea_agent,
    expected_output="A structured list of suggested projects.",
    output_schema=ProjectSuggestionsOutput
)

# --- Compose Crew and Run Sequentially ---

crew = Crew(
    agents=[learning_material_agent, quiz_creator_agent, project_idea_agent],
    tasks=[learning_material_task, quiz_task, project_task],
    process=Process.sequential,
    verbose=True
)

results = crew.kickoff()


In [ ]:
1